## Transformer Based Embedding Generation

AllMiniLmL6V2

In [ ]:
#Configuration

MODEL_ID   = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_NAME = "all-minilm-l6-v2"   # used as key in embeddings dict + filename

# Paths
CHUNKS_PATH    = "../data/chunks/enriched_chunks.json"
QA_PAIRS_PATH  = "../data/qa_pairs/qa_pairs.json"
EMBEDDINGS_DIR = "../data/embeddings"
EMBEDDINGS_PATH = f"{EMBEDDINGS_DIR}/embeddings.json"

# Embedding config
BATCH_SIZE     = 64     # reduce if OOM
MAX_SEQ_LENGTH = 256    # matches chunk token limit — override per model if needed

# Some models need a query prompt prefix for asymmetric retrieval
# Set to None if model does not use prompts (e.g. MiniLM, MPNet)
# Set to prompt name string for models that do (e.g. "query" for Qwen3-Embedding)
QUERY_PROMPT_NAME  = None   # e.g. "query" for Qwen3-Embedding
PASSAGE_PROMPT_NAME = None  # e.g. "passage" for Qwen3-Embedding

print(f"Model     : {MODEL_ID}")
print(f"Model key : {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE}")

Model     : sentence-transformers/all-MiniLM-L6-v2
Model key : all-minilm-l6-v2
Batch size: 64


In [3]:
#Imports and Setup
import os
import json
import torch
import gc
from tqdm import tqdm
from pathlib import Path
from sentence_transformers import SentenceTransformer

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM available: 8.6 GB


In [7]:
#Load Model
print(f"Loading {MODEL_ID}...")

model = SentenceTransformer(MODEL_ID, device=device)

# Override max sequence length if configured
if MAX_SEQ_LENGTH and hasattr(model, "max_seq_length"):
    model.max_seq_length = MAX_SEQ_LENGTH

print(f"Model loaded.")
print(f"Max sequence length: {model.max_seq_length}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading sentence-transformers/all-MiniLM-L6-v2...
Model loaded.
Max sequence length: 256
Embedding dimension: 384


In [8]:
#Load Data
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

with open(QA_PAIRS_PATH, "r", encoding="utf-8") as f:
    all_qa_pairs = json.load(f)

print(f"Chunks loaded   : {len(all_chunks)}")
print(f"QA pairs loaded : {len(all_qa_pairs)}")
print(f"\nChunk sample:")
print(json.dumps({k: v for k, v in all_chunks[0].items() if k != "figures_on_page"}, indent=2))
print(f"\nQA pair sample:")
print(json.dumps(all_qa_pairs[0], indent=2))

Chunks loaded   : 3055
QA pairs loaded : 5943

Chunk sample:
{
  "chunk_id": "page_1_chunk_0",
  "text": "N E X A\nGRAND VITARA\nOWNER'S MANUAL\nFor vehicle fitted with CNG fuel system, use Grand Vitara CNG Supplementary Owner's Manual\n\u00b7 Refer for vehicle usage at all times\n\u00b7 Contains Warranty policy and Important Information On Safety, Operation & Maintenance\nPart No. 99011M76T11-74W March, 2026 ENG",
  "contextualized_text": "N E X A\nGRAND VITARA\nOWNER'S MANUAL\nFor vehicle fitted with CNG fuel system, use Grand Vitara CNG Supplementary Owner's Manual\n\u00b7 Refer for vehicle usage at all times\n\u00b7 Contains Warranty policy and Important Information On Safety, Operation & Maintenance\nPart No. 99011M76T11-74W March, 2026 ENG",
  "page_no": 1,
  "heading": "N E X A",
  "token_count": 74
}

QA pair sample:
{
  "qa_id": "page_2_chunk_1_q0",
  "chunk_id": "page_2_chunk_1",
  "page_no": 2,
  "question": "Can you tell me where the rear tailgate is located on the Grand Vi

In [9]:
# Load existing embeddings file if it exists
# This allows adding a new model without re-embedding with previous models

if Path(EMBEDDINGS_PATH).exists():
    print(f"Existing embeddings file found — loading...")
    with open(EMBEDDINGS_PATH, "r", encoding="utf-8") as f:
        embeddings_store = json.load(f)

    existing_chunk_ids    = {c["chunk_id"] for c in embeddings_store["chunks"]}
    existing_qa_ids       = {q["qa_id"]    for q in embeddings_store["qa_pairs"]}
    already_has_model_chunks = any(
        MODEL_NAME in c["embeddings"]
        for c in embeddings_store["chunks"]
    )

    print(f"Chunks in store  : {len(embeddings_store['chunks'])}")
    print(f"QA pairs in store: {len(embeddings_store['qa_pairs'])}")
    print(f"Model '{MODEL_NAME}' already embedded: {already_has_model_chunks}")

else:
    print("No existing embeddings file — creating fresh store...")

    # Build store structure from chunks and qa_pairs
    # Each entry has empty embeddings dict — models fill it in
    embeddings_store = {
        "chunks"  : [
            {
                "chunk_id"   : c["chunk_id"],
                "page_no"    : c.get("page_no"),
                "text"       : c.get("contextualized_text") or c.get("text", ""),
                "heading"    : c.get("heading"),
                "is_figure"  : c.get("is_figure_chunk", False),
                "embeddings" : {},
            }
            for c in all_chunks
        ],
        "qa_pairs": [
            {
                "qa_id"         : q["qa_id"],
                "chunk_id"      : q["chunk_id"],
                "page_no"       : q.get("page_no"),
                "question"      : q["question"],
                "question_type" : q.get("question_type"),
                "embeddings"    : {},
            }
            for q in all_qa_pairs
        ],
    }

    print(f"Store initialized: {len(embeddings_store['chunks'])} chunks, "
          f"{len(embeddings_store['qa_pairs'])} QA pairs")

No existing embeddings file — creating fresh store...
Store initialized: 3055 chunks, 5943 QA pairs


In [ ]:
#Embed Chunks

def embed_batch(texts: list[str], prompt_name: str | None = None) -> list[list[float]]:
    """
    Embed a batch of texts. Returns list of embedding vectors.
    Handles prompt_name for models that support asymmetric retrieval.
    """
    kwargs = {"show_progress_bar": False, "convert_to_numpy": True}
    if prompt_name and prompt_name in (model.prompts or {}):
        kwargs["prompt_name"] = prompt_name
    return model.encode(texts, **kwargs).tolist()


# Find chunks that don't have this model's embedding yet
chunks_to_embed = [
    (i, c) for i, c in enumerate(embeddings_store["chunks"])
    if MODEL_NAME not in c["embeddings"]
]

print(f"Chunks to embed: {len(chunks_to_embed)} "
      f"({len(embeddings_store['chunks']) - len(chunks_to_embed)} already done)")

if chunks_to_embed:
    for batch_start in tqdm(
        range(0, len(chunks_to_embed), BATCH_SIZE),
        desc="Embedding chunks"
    ):
        batch        = chunks_to_embed[batch_start:batch_start + BATCH_SIZE]
        batch_idxs   = [b[0] for b in batch]
        batch_texts  = [b[1]["text"] for b in batch]

        try:
            embeddings = embed_batch(batch_texts, prompt_name=PASSAGE_PROMPT_NAME)
            for idx, embedding in zip(batch_idxs, embeddings):
                embeddings_store["chunks"][idx]["embeddings"][MODEL_NAME] = embedding

        except Exception as e:
            print(f"  Batch {batch_start}-{batch_start+BATCH_SIZE} failed: {e}")
            continue

        # Save every 500 chunks to avoid losing progress
        if (batch_start // BATCH_SIZE) % (500 // BATCH_SIZE + 1) == 0:
            with open(EMBEDDINGS_PATH, "w", encoding="utf-8") as f:
                json.dump(embeddings_store, f)

        # Free GPU cache periodically
        if device == "cuda" and (batch_start // BATCH_SIZE) % 10 == 0:
            torch.cuda.empty_cache()

    # Save after all chunks done
    with open(EMBEDDINGS_PATH, "w", encoding="utf-8") as f:
        json.dump(embeddings_store, f)

    print(f"Chunks embedded and saved.")
else:
    print("All chunks already embedded for this model — skipping.")

Chunks to embed: 3055 (0 already done)


Embedding chunks: 100%|██████████| 48/48 [00:07<00:00,  6.37it/s]


Chunks embedded and saved.


In [11]:
#Embed QA Pairs
# Find QA pairs that don't have this model's embedding yet
qa_to_embed = [
    (i, q) for i, q in enumerate(embeddings_store["qa_pairs"])
    if MODEL_NAME not in q["embeddings"]
]

print(f"QA pairs to embed: {len(qa_to_embed)} "
      f"({len(embeddings_store['qa_pairs']) - len(qa_to_embed)} already done)")

if qa_to_embed:
    for batch_start in tqdm(
        range(0, len(qa_to_embed), BATCH_SIZE),
        desc="Embedding questions"
    ):
        batch        = qa_to_embed[batch_start:batch_start + BATCH_SIZE]
        batch_idxs   = [b[0] for b in batch]
        batch_texts  = [b[1]["question"] for b in batch]

        try:
            embeddings = embed_batch(batch_texts, prompt_name=QUERY_PROMPT_NAME)
            for idx, embedding in zip(batch_idxs, embeddings):
                embeddings_store["qa_pairs"][idx]["embeddings"][MODEL_NAME] = embedding

        except Exception as e:
            print(f"  Batch {batch_start}-{batch_start+BATCH_SIZE} failed: {e}")
            continue

        if (batch_start // BATCH_SIZE) % (500 // BATCH_SIZE + 1) == 0:
            with open(EMBEDDINGS_PATH, "w", encoding="utf-8") as f:
                json.dump(embeddings_store, f)

        if device == "cuda" and (batch_start // BATCH_SIZE) % 10 == 0:
            torch.cuda.empty_cache()

    with open(EMBEDDINGS_PATH, "w", encoding="utf-8") as f:
        json.dump(embeddings_store, f)

    print(f"QA pairs embedded and saved.")
else:
    print("All QA pairs already embedded for this model — skipping.")

QA pairs to embed: 5943 (0 already done)


Embedding questions: 100%|██████████| 93/93 [00:38<00:00,  2.39it/s]


QA pairs embedded and saved.


In [12]:
# Verify completeness
chunks_missing   = sum(1 for c in embeddings_store["chunks"]   if MODEL_NAME not in c["embeddings"])
qa_missing       = sum(1 for q in embeddings_store["qa_pairs"] if MODEL_NAME not in q["embeddings"])
models_in_store  = set()
for c in embeddings_store["chunks"]:
    models_in_store.update(c["embeddings"].keys())

print(f"Embeddings file : {EMBEDDINGS_PATH}")
print(f"Models in store : {sorted(models_in_store)}")
print(f"Chunks total    : {len(embeddings_store['chunks'])}")
print(f"Chunks missing  : {chunks_missing}")
print(f"QA pairs total  : {len(embeddings_store['qa_pairs'])}")
print(f"QA missing      : {qa_missing}")
print(f"\nEmbedding dim   : {len(embeddings_store['chunks'][0]['embeddings'].get(MODEL_NAME, []))}")

if chunks_missing == 0 and qa_missing == 0:
    print(f"\n✓ Model '{MODEL_NAME}' fully embedded — ready for benchmarking")
else:
    print(f"\n✗ Re-run cells 5 and 6 to complete missing embeddings")

Embeddings file : ../data/embeddings/all-minilm-l6-v2_embeddings.json
Models in store : ['all-minilm-l6-v2']
Chunks total    : 3055
Chunks missing  : 0
QA pairs total  : 5943
QA missing      : 0

Embedding dim   : 384

✓ Model 'all-minilm-l6-v2' fully embedded — ready for benchmarking


In [13]:
#Unload Model
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"VRAM freed")
print("Model unloaded")

VRAM freed
Model unloaded
